# Per-cell graph phenotype — modality atlas across the blinatumomab cohort

A descriptive atlas of interpretable **per-cell graph modalities**, computed by
`graph_phenotype.py` (see `run_graph_phenotype.lsf`). Each section owns one **family** of
modalities and simply **presents the comparisons** — the biology is read off by the analyst, so
there is no significance testing, hit-calling, or depth-gating here.

## Graph model (read once)

Each cell (`component`) is one **bipartite** graph: side-1 nodes = `umi1` (the A-pixels), side-2
nodes = `umi2` (the B-pixels); edges only cross sides. Degree (Section 2) and topology
(Section 3) treat this as a **plain graph — every node is a node, sides are ignored**. Topology
metrics are also scored against a **degree-preserving edge-swap null** (rewire edges keeping
both side degree sequences fixed): `{feat}_ratio = observed / null-mean` (≈ 1 ⇒ fully explained
by the degree sequence). Because the graph is strictly bipartite it has no triangles, so
clustering / transitivity are identically 0 and are not reported.

## Grouping axes

- **`cell_system`** (up to 4 columns): `healthy B + healthy T`, `NALM-6 + healthy T`,
  `patient B + patient T`, `NALM-6 + patient T` — the last is incomplete (no 48h-Mock arm), so
  one x-slot stays empty.
- **The 4 conditions on every x-axis** = `condition × time`: `Mock_6h`, `Blinatumomab_6h`,
  `Mock_48h`, `Blinatumomab_48h` (blue = Mock, orange = Blinatumomab; light = 6h, dark = 48h).
- **`cell_type_annot`**: only **CD8** and **B** — the graph phenotype was run with
  `--cell-types CD8,B` at **native depth**.

## Three views in every section

- **View A — entire cohort**: pooled distribution of each modality over all cells.
- **View B — systems × conditions**: the 4 conditions on x, one column per system (all cell
  types pooled). This is the core comparison.
- **View C — per cell type**: View B repeated for CD8, then B.

## Sections

**0** submit the compute (unchanged) · **1** size · **2** degree · **3** topology · **4** path ·
**5** patches (LNE). Depth (`n_edges`) is itself a size modality (Section 1) and is confounded
with both `condition` and `cell_system` — it is kept in view here, not corrected.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "/home/projects/nyosef/zvise/PixelGen/PixelGen")
sys.path.insert(0, "/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data")
import numpy as np, pandas as pd, scanpy as sc
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

try:
    from nalm_utils import display_name
except Exception:
    display_name = None

def lab(c):
    """Pretty label for a modality column, falling back to the raw name."""
    if display_name is None:
        return c
    try:
        s = display_name(c)
        return s if isinstance(s, str) and s else c
    except Exception:
        return c

sc.set_figure_params(dpi=100, frameon=False)
sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (5, 3), "figure.max_open_warning": 0})

BASE      = Path("/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data")
CACHE     = BASE / "cache"
GP        = BASE / "results" / "graph_phenotype"
ANNOTATED = CACHE / "adata_cytovi_annotated_compat.h5ad"

# ---- grouping axes (Views B/C) --------------------------------------------------------------
SYSTEM_ORDER   = ["healthy B + healthy T", "NALM-6 + healthy T",
                  "patient B + patient T", "NALM-6 + patient T"]   # last incomplete (no 48h Mock)
GROUP_ORDER    = ["Mock_6h", "Blinatumomab_6h", "Mock_48h", "Blinatumomab_48h"]  # condition x time
CELLTYPE_ORDER = ["CD8", "B"]                                      # graph phenotype: CD8+B only

# blue = Mock, orange = Blinatumomab; light = 6h, dark = 48h
GROUP_PALETTE  = {"Mock_6h": "#9ecae1", "Blinatumomab_6h": "#fdae6b",
                  "Mock_48h": "#3182bd", "Blinatumomab_48h": "#e6550d"}
SYSTEM_PALETTE = {"healthy B + healthy T": "#2ca02c", "NALM-6 + healthy T": "#9467bd",
                  "patient B + patient T": "#1f77b4", "NALM-6 + patient T": "#d62728"}

# count-like modalities -> log x in the cohort-overview histograms (View A)
LOG_FEATS = {"n_umi", "n_edges", "n_edges_used", "n_edges_full", "n_nodes", "n1", "n2",
             "n_possible_edges", "proj_n_edges", "proj_degree_max", "u1_degree_max",
             "u2_degree_max", "lne_marker_nodes", "act_lne_marker_nodes",
             "inh_lne_marker_nodes", "reads_per_umi"}

SIZE_COVARIATES = ["n_umi", "n_edges", "n_umi1", "n_umi2", "reads_per_umi",
                   "n_antibodies", "tau", "isotype_fraction"]

## Step 0 — submit the compute

`graph_phenotype.py` reads the `.pxl` edgelists, so it runs on the cluster, not in this kernel.
Native depth, 20 nulls, LNE (`local_proximity`, k=3) → CD8+B finishes in minutes on 16 workers
per sample. The `(A+I)^k` reachability at k=3 is the new cost; if a job OOMs (`gphen-*.err`),
drop `--n-workers` or fall back to k=2 in `lne_features`.

The next cell submits one LSF job per sample. It is **idempotent** — it skips samples whose
parquet already exists — so re-run it freely to top up after a failure. Flip `SUBMIT = False`
to print the `bsub` lines without firing them.

⚠️ **Switching block 6 to LNE changed the emitted columns** (`lne_*` replace `n_patches` /
`patch_*` / `largest_patch`). Because submit skips samples whose parquet exists, the existing
`*_graph_phenotype.parquet` files must be **deleted or moved first** for the `lne_*` columns to
regenerate — otherwise the load cell finds the old patch columns and `PATCH_FEATS` comes back
empty.

`EDGE_BUDGETS="0"` (native depth) is set in the `.lsf` and should stay there: non-zero budgets
exist only to reproduce the depth curve as a diagnostic, and make the confound worse, not better.

In [ ]:
# Submit one LSF job per sample (mirrors final_sweep_benchmark.ipynb's submit cell).
# Idempotent on BOTH the output parquet and the queue: re-running while jobs are still
# pending would otherwise resubmit all of them, since no parquet exists yet.
# Flip SUBMIT to False to dry-run.
import subprocess

SUBMIT = True                      # <-- False prints the bsub lines without firing them
CELL_TYPES = "CD8,B"               # B cells are the patch control; "" = all cell types
SAMPLES = ["S001", "S002", "S003", "S004", "S005", "S006", "S007", "S008",
           "S009", "S010", "S011", "S012", "S013", "S014", "S016"]  # S015 missing from delivery

# Where to run. `long` is free but priority 71 and preemptable, and can be jammed hard enough
# that NOTHING places at any size — on 2026-07-15 its host group (public_cpu_hosts) was 51/62
# closed_Full, 9 closed_Busy, 2 ok with ~1 free slot between them, and 16/4/1-thread asks all
# sat pending alike. gsla_high_gpu dispatched instantly (dgn nodes had 27-63 free cores).
# Check `bhosts` on the host group before assuming the ask is too big — the pool may be empty.
QUEUE = "gpu"                      # "long" (free, often jammed) | "gpu" (instant, costs a GPU)

# 16 threads / 32G. Memory drops (a running job peaks at 6-7.3G, so the old 64G was ~10x over)
# but the cores stay: gphen hits 2.86x parallelism on 4 threads (72% efficiency), so it scales
# well and wants them — at 4 workers a sample takes ~39 min (3.3 s/cell x ~707 cells) vs ~13
# at 16. Do NOT size this off chunglu's 3.4x-on-16: different script, different scaling.
# Keep --n-workers in step with the thread count (the .lsf default is 16).
QUEUE_ARGS = {
    "long": ["-q", "long"],
    # graph_phenotype.py never touches CUDA (igraph/numpy/scipy only), so j_exclusive=no:
    # this queue requires holding a GPU, but we needn't lock colleagues out of a card we
    # leave 100% idle. Consumes the group's ngpus_phys quota (10, shared with the lab's
    # notebook kernels) — `blimits -u $USER` to check. 15 jobs saturate it, so only ~6 run
    # at once; the rest queue behind them. Fine as a burst, not as a default.
    "gpu":  ["-q", "gsla_high_gpu", "-gpu", "num=1:j_exclusive=no"],
}[QUEUE]

LSF  = BASE / "run_graph_phenotype.lsf"
LOGS = BASE / "results" / "logs"
LOGS.mkdir(parents=True, exist_ok=True)
GP.mkdir(parents=True, exist_ok=True)


def queued_samples():
    """Samples with a gphen job already PEND/RUN, so we don't submit a second copy."""
    r = subprocess.run(["bjobs", "-noheader", "-J", "gphen-*", "-o", "job_name stat"],
                       capture_output=True, text=True)
    return {ln.split()[0].removeprefix("gphen-") for ln in r.stdout.splitlines()
            if len(ln.split()) == 2 and ln.split()[1] in ("PEND", "RUN")}

in_queue = queued_samples()

n_sub = n_skip = n_queued = n_missing = 0
for s in SAMPLES:
    if (GP / f"{s}_graph_phenotype.parquet").exists():
        n_skip += 1
        continue
    if s in in_queue:
        n_queued += 1
        continue
    pxl = BASE / "results" / s / "layout" / "layout" / f"{s}.layout.pxl"
    if not pxl.exists():
        print(f"  {s}: no .pxl — skipping ({pxl})")
        n_missing += 1
        continue
    # NOTE the .pxl is a DuckDB database and pixelator opens it WITHOUT read_only, taking an
    # exclusive write lock. A notebook kernel holding that sample's dataset makes this job die
    # with "Could not set lock ... PID 0" (PID 0 = holder is on another host). If exactly one
    # sample fails that way, free the kernel that has it open, then re-run this cell.
    #
    # Resources go on the command line: #BSUB directives inside the script are only parsed
    # when it is fed on stdin, and we need `-- bash script --args` to pass --cell-types
    # (LSF's -env splits on commas, so -env "CELL_TYPES=CD8,B" would read B as a var name).
    cmd = (["bsub", "-J", f"gphen-{s}"] + QUEUE_ARGS +
           ["-R", "rusage[mem=32G]", "-R", "affinity[thread*16]",
            "-o", str(LOGS / f"gphen-{s}.out"), "-e", str(LOGS / f"gphen-{s}.err"),
            "--", "bash", str(LSF), "--sample", s])
    if CELL_TYPES:
        cmd += ["--cell-types", CELL_TYPES]
    if SUBMIT:
        subprocess.run(cmd, check=True)
    else:
        print(" ".join(cmd))
    n_sub += 1

print(f"\nqueue={QUEUE} | submitted {n_sub} | already done {n_skip} | already queued {n_queued} "
      f"| no pxl {n_missing}{'  (DRY RUN — set SUBMIT=True)' if not SUBMIT and n_sub else ''}")

In [ ]:
# Poll until the gphen jobs finish. Safe to re-run / interrupt — it only reads state.
import time

WAIT = False                        # <-- False to just print a one-shot status
POLL_SECONDS, TIMEOUT_MINUTES = 30, 60

def gphen_status():
    """(n_running, n_done_parquets, per-state counts) for the gphen jobs."""
    r = subprocess.run(["bjobs", "-noheader", "-J", "gphen-*", "-o", "stat"],
                       capture_output=True, text=True)
    states = [l.strip() for l in r.stdout.splitlines() if l.strip()]
    counts = {s: states.count(s) for s in set(states)}
    done = len(list(GP.glob("*_graph_phenotype.parquet")))
    return len(states), done, counts

t0 = time.time()
while True:
    n_active, n_done, counts = gphen_status()
    print(f"[{time.strftime('%H:%M:%S')}] active={n_active} {counts or ''} | "
          f"parquets written={n_done}/{len(SAMPLES)}", flush=True)
    if not WAIT or n_active == 0:
        break
    if time.time() - t0 > TIMEOUT_MINUTES * 60:
        print(f"still running after {TIMEOUT_MINUTES} min — check {LOGS}/gphen-*.err")
        break
    time.sleep(POLL_SECONDS)

if n_done == 0:
    print(f"\nNo parquets yet. If jobs already exited, read the errors:\n"
          f"  tail -20 {LOGS}/gphen-*.err")
else:
    missing = [s for s in SAMPLES if not (GP / f"{s}_graph_phenotype.parquet").exists()]
    print(f"\ndone. missing: {missing if missing else 'none'}")

In [ ]:
adata = sc.read_h5ad(ANNOTATED)
print("adata:", adata.shape)

files = sorted(GP.glob("*_graph_phenotype.parquet"))
assert files, f"No parquets in {GP} — run run_graph_phenotype.lsf first (Section 0)."
gp = pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)
print(f"graph-phenotype rows: {len(gp):,} ({gp.component.nunique():,} cells "
      f"x {gp.edge_budget.nunique()} budget) from {len(files)} samples")

# obs and gp both carry n_edges with DIFFERENT meaning: gp.n_edges = edges used in the graph,
# obs.n_edges = the cell's native depth. Keep both — rename gp's to n_edges_used.
gp = gp.rename(columns={"n_edges": "n_edges_used"}).drop(columns=["sample"])

In [ ]:
OBS_COLS = ["cell_type_annot", "cell_system", "condition", "time", "sample"] + SIZE_COVARIATES

# One frame, native depth (edge_budget == 0 => one row per cell).
cells = (gp[gp.edge_budget == 0].set_index("component")
           .join(adata.obs[OBS_COLS], how="inner"))

# derived: number of possible cross-side edges (bipartite) = n1 * n2.
cells["n_possible_edges"] = cells.n1 * cells.n2

# time x condition -> the 4 groups on every x-axis; ordered so panels read Mock/Blina x 6h/48h.
cells["group"] = pd.Categorical(cells.condition.astype(str) + "_" + cells.time.astype(str),
                                categories=GROUP_ORDER, ordered=True)
cells["cell_system"] = pd.Categorical(cells.cell_system, categories=SYSTEM_ORDER, ordered=True)
cells["cell_type_annot"] = pd.Categorical(cells.cell_type_annot,
                                          categories=CELLTYPE_ORDER, ordered=True)

def present(names):
    """Keep only columns that exist in `cells` (families are declared, then intersected)."""
    return [c for c in names if c in cells.columns]

# ---- modality families (Sections 1-5) -------------------------------------------------------
SIZE_FEATS = present(["n_nodes", "n_edges_used", "n_possible_edges", "bip_density"])

# combined-side node degree (regardless of side); added by graph_phenotype.py -> re-run Section 0.
DEGREE_FEATS = present(["deg_mean", "deg_std", "deg_skew", "deg_gini", "deg_kappa"])

# whole-graph topology (raw graph, sides ignored); g_* columns -> re-run Section 0.
TOPO_FEATS = present(["g_density", "g_n_components", "g_largest_cc_frac",
                      "g_average_k_core", "g_degree_assortativity"])
TOPO_RATIO = present([f"{f}_ratio" for f in TOPO_FEATS])          # g_density has no null

PATH_FEATS  = present(["g_avg_path_length", "g_diameter"])        # subsampled largest component
PATCH_FEATS = present(["lne_mean", "lne_p90", "lne_max", "lne_marker_nodes"])

print(f"cells: {cells.shape[0]:,} | cell types {cells.cell_type_annot.value_counts().to_dict()}")
print(f"systems present: {[s for s in SYSTEM_ORDER if (cells.cell_system == s).any()]}")
print(f"families -> size {len(SIZE_FEATS)} | degree {len(DEGREE_FEATS)} | "
      f"topology {len(TOPO_FEATS)} (+{len(TOPO_RATIO)} ratio) | "
      f"path {len(PATH_FEATS)} | patches {len(PATCH_FEATS)}")
if not (DEGREE_FEATS and TOPO_FEATS and PATH_FEATS):
    print("NOTE: deg_*/g_* columns missing — delete results/graph_phenotype/*.parquet and re-run Section 0.")

In [ ]:
# Reusable comparison plots — kept DRY so each section is just calls.
# Titles are descriptive labels (modality / view), never findings.

def _panel_box(ax, sub, feat):
    """Boxplot of `feat` vs the 4 condition x time groups on one axis (+ a light cell strip)."""
    data, cols, pos = [], [], []
    for k, g in enumerate(GROUP_ORDER):
        v = sub.loc[sub.group == g, feat].replace([np.inf, -np.inf], np.nan).dropna().values
        if len(v):
            data.append(v); cols.append(GROUP_PALETTE[g]); pos.append(k)
    if data:
        bp = ax.boxplot(data, positions=pos, widths=.6, patch_artist=True, showfliers=False,
                        medianprops=dict(color="black", lw=1),
                        whiskerprops=dict(color="0.5", lw=.6), capprops=dict(color="0.5", lw=.6),
                        boxprops=dict(edgecolor="0.5", lw=.5))
        for patch, c in zip(bp["boxes"], cols):
            patch.set_facecolor(c); patch.set_alpha(.85)
        rng = np.random.default_rng(0)
        for p, v in zip(pos, data):
            vv = v if len(v) <= 300 else rng.choice(v, 300, replace=False)
            ax.scatter(rng.normal(p, .07, len(vv)), vv, s=1.3, color="0.2", alpha=.18, lw=0)
    # Set ticks LAST: ax.boxplot() overwrites xticks with numeric positions, so the real group
    # names must be applied after it, else the axis shows 0/1/2/3.
    ax.set_xlim(-.6, len(GROUP_ORDER) - .4)
    ax.set_xticks(range(len(GROUP_ORDER)))
    ax.set_xticklabels(GROUP_ORDER, rotation=45, ha="right", fontsize=6)
    ax.tick_params(axis="y", labelsize=6)


def grid_by_system(feats, df=None, systems=None, title=None):
    """View B/C — rows = modalities, cols = cell_system; each panel = the 4 groups on x.

    y is shared per row so systems are comparable; empty (system, group) slots stay blank.
    """
    df = cells if df is None else df
    feats = [f for f in feats if f in df.columns]
    systems = [s for s in (systems or SYSTEM_ORDER) if (df.cell_system == s).any()]
    if not feats or not systems:
        print("  (nothing to plot)"); return
    nr, nc = len(feats), len(systems)
    fig, axes = plt.subplots(nr, nc, figsize=(2.9 * nc, 1.9 * nr), squeeze=False, sharey="row")
    for i, feat in enumerate(feats):
        for j, s in enumerate(systems):
            ax = axes[i][j]
            _panel_box(ax, df[df.cell_system == s], feat)
            if i == 0:
                ax.set_title(s, fontsize=8)
            ax.set_ylabel(lab(feat) if j == 0 else "", fontsize=7)
        vals = (df.loc[df.cell_system.isin(systems), feat]
                  .replace([np.inf, -np.inf], np.nan).dropna())
        if len(vals) > 10:
            lo, hi = np.nanpercentile(vals, [1, 99])
            if hi > lo:
                axes[i][0].set_ylim(lo - (hi - lo) * .08, hi + (hi - lo) * .08)
    if title:
        fig.suptitle(title, fontsize=11, y=1.003, fontweight="bold")
    fig.tight_layout(); plt.show()


def family_overview(feats, df=None, ncols=4, title=None):
    """View A — pooled distribution of every modality over the whole cohort."""
    df = cells if df is None else df
    feats = [f for f in feats if f in df.columns]
    if not feats:
        print("  (no features)"); return
    ncols = min(ncols, len(feats)); nrows = int(np.ceil(len(feats) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 2.1 * nrows), squeeze=False)
    axes = axes.ravel()
    for k, feat in enumerate(feats):
        ax = axes[k]
        v = df[feat].replace([np.inf, -np.inf], np.nan).dropna()
        if v.empty or v.nunique() < 2:
            ax.set_title(f"{lab(feat)} (n/a)", fontsize=7); ax.set_axis_off(); continue
        if feat in LOG_FEATS and v.min() > 0:
            ax.hist(v, bins=np.logspace(np.log10(v.min()), np.log10(v.max()), 40),
                    color="#4c72b0", alpha=.85); ax.set_xscale("log")
        else:
            lo, hi = np.nanpercentile(v, [0.5, 99.5])
            ax.hist(v, bins=40, range=(lo, hi) if hi > lo else None, color="#4c72b0", alpha=.85)
        ax.axvline(v.median(), color="crimson", lw=1, ls="--")
        ax.set_title(lab(feat), fontsize=8); ax.set_yticks([]); ax.tick_params(labelsize=6)
    for ax in axes[len(feats):]:
        ax.set_axis_off()
    if title:
        fig.suptitle(title, fontsize=11, y=1.002, fontweight="bold")
    fig.tight_layout(); plt.show()

## Section 1 — SIZE

The raw scale of each cell's bipartite graph: how many nodes and edges it has, how many
cross-side edges are possible, and what fraction of those possible edges actually exist. All
four are exact counts from the edgelist (`compute_topology`); at edge-budget 0 they are the
cell's native depth.

| modality | definition |
|---|---|
| `n_nodes` | number of nodes — distinct nodes touched by edges, `np.unique(edges).size` |
| `n_edges_used` | number of edges — `G.ecount()` after de-dup and self-loop removal |
| `n_possible_edges` | number of possible cross-side edges = `n1 · n2` (side-1 × side-2 node counts) |
| `bip_density` | proportion of possible edges that exist = `n_edges_used / (n1 · n2)`, in [0, 1] |

In [ ]:
# View A — entire cohort: pooled distribution of every size modality.
family_overview(SIZE_FEATS, title="Section 1 · SIZE — cohort distributions")

In [ ]:
# View B — systems x conditions (all cell types pooled): 4 conditions on x, one column per system.
grid_by_system(SIZE_FEATS, title="SIZE — systems x conditions")

In [ ]:
# View C — per cell type (CD8, then B).
for ct in CELLTYPE_ORDER:
    grid_by_system(SIZE_FEATS, df=cells[cells.cell_type_annot == ct], title=f"SIZE — {ct} cells")

## Section 2 — DEGREE

Node **degree** counted over the whole graph, **regardless of side**: every node — side-1
(A-pixel) and side-2 (B-pixel) alike — contributes its degree = number of incident edges. Five
summary statistics per cell describe the shape of that per-cell degree distribution, then the
pooled degree distribution itself over a sample of cells.

| modality | definition |
|---|---|
| `deg_mean` | mean node degree over all nodes (= `2 · n_edges / n_nodes`) |
| `deg_std` | population std (ddof=0) of node degree |
| `deg_skew` | Fisher–Pearson skewness of the degree distribution (`nan` if ≤ 2 nodes) |
| `deg_gini` | Gini coefficient of the degree distribution, in [0, 1] |
| `deg_kappa` | degree heterogeneity κ = `⟨d²⟩ / ⟨d⟩²` (= 1 for a regular graph, grows with hubs) |

> These `deg_*` columns are added by `graph_phenotype.py`. **Delete the old parquets and re-run
> Section 0** to regenerate them; until then `present()` drops them and Views A–C are empty.

In [ ]:
# View A — entire cohort.
family_overview(DEGREE_FEATS, title="Section 2 · DEGREE — cohort distributions")

In [ ]:
# View B — systems x conditions (all cell types pooled).
grid_by_system(DEGREE_FEATS, title="DEGREE — systems x conditions")

In [ ]:
# View C — per cell type (CD8, then B).
for ct in CELLTYPE_ORDER:
    grid_by_system(DEGREE_FEATS, df=cells[cells.cell_type_annot == ct], title=f"DEGREE — {ct} cells")

### Degree distribution (sampled cells)

The pooled node-degree distribution, drawn from a stratified random sample of cells (the five
statistics above summarise it per cell; this shows the raw shape). Loads each sampled cell's
graph from its `.pxl` — see the lock caveat in the cell.

In [ ]:
# Degree distribution — pool node degrees from a random SAMPLE of cells (loading every cell's
# graph is too heavy). Reads the .pxl edgelists via pixelator, so run this only when no cluster
# job is touching these samples: the .pxl is a DuckDB file held under an exclusive write lock.
from chunglu_triplets import load_cell

DEG_SAMPLE_N = 80   # cells sampled per cell type

def sample_node_degrees(df, n_per_type=DEG_SAMPLE_N, seed=0):
    """Load a stratified sample of cells and return long-form (component, degree, groups)."""
    from pixelator import read_pna
    # degrees only need graph structure; any consistent marker->index map lets load_cell run.
    marker_to_idx = {m: i for i, m in enumerate(adata.var_names)}
    picks = (df.reset_index()
               .groupby("cell_type_annot", observed=True, group_keys=False)
               .apply(lambda g: g.sample(min(n_per_type, len(g)), random_state=seed)))
    rows = []
    for s, sub in picks.groupby("sample", observed=True):
        pxl = BASE / "results" / s / "layout" / "layout" / f"{s}.layout.pxl"
        if not pxl.exists():
            print(f"  {s}: no .pxl, skipped"); continue
        pg = read_pna([str(pxl)])
        for _, r in sub.iterrows():
            el = (pg.filter(components=[r["component"]]).edgelist().to_df()
                    [["umi1", "umi2", "marker_1", "marker_2"]])
            edges, labels, side, n = load_cell(el, marker_to_idx)
            if edges.shape[0] == 0:
                continue
            deg = np.bincount(edges.ravel()); deg = deg[deg > 0]
            rows.append(pd.DataFrame({"degree": deg, "cell_type_annot": r["cell_type_annot"],
                                      "cell_system": r["cell_system"], "component": r["component"]}))
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def _ccdf(d):
    """Complementary CDF P(X >= x): straight line on log-log <=> power law."""
    x = np.sort(np.asarray(d, dtype=float))
    y = 1.0 - np.arange(len(x)) / len(x)   # fraction of nodes with degree >= x
    return x, y

deg_long = sample_node_degrees(cells)
if deg_long.empty:
    print("no degrees loaded (missing .pxl?) — skipping distribution plot")
else:
    print(f"pooled {len(deg_long):,} node degrees from {deg_long.component.nunique()} cells")
    bins = np.logspace(0, np.log10(max(int(deg_long.degree.max()), 2)), 40)
    # Row 0 = log-log density histogram; Row 1 = CCDF (power-law view). Cols = by cell type / system.
    fig, axes = plt.subplots(2, 2, figsize=(9, 6))
    for ct, c in zip(CELLTYPE_ORDER, ["#1f77b4", "#d62728"]):
        d = deg_long.loc[deg_long.cell_type_annot == ct, "degree"]
        if len(d):
            axes[0, 0].hist(d, bins=bins, histtype="step", lw=1.6, color=c, density=True,
                            label=f"{ct} (n={len(d):,})")
            x, y = _ccdf(d)
            axes[1, 0].plot(x, y, lw=1.6, color=c, label=ct)
    for sysname in [s for s in SYSTEM_ORDER if (deg_long.cell_system == s).any()]:
        d = deg_long.loc[deg_long.cell_system == sysname, "degree"]
        axes[0, 1].hist(d, bins=bins, histtype="step", lw=1.3, density=True,
                        color=SYSTEM_PALETTE[sysname], label=sysname)
        x, y = _ccdf(d)
        axes[1, 1].plot(x, y, lw=1.3, color=SYSTEM_PALETTE[sysname], label=sysname)

    axes[0, 0].set_title("degree distribution by cell type", fontsize=9)
    axes[0, 1].set_title("degree distribution by system", fontsize=9)
    axes[1, 0].set_title("CCDF by cell type (power-law check)", fontsize=9)
    axes[1, 1].set_title("CCDF by system (power-law check)", fontsize=9)
    for ax in axes[0]:
        ax.set_ylabel("density")
    for ax in axes[1]:
        ax.set_ylabel("P(degree ≥ x)")
    for ax in axes.ravel():
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel("node degree")
    axes[0, 0].legend(fontsize=7); axes[0, 1].legend(fontsize=6)
    axes[1, 0].legend(fontsize=7); axes[1, 1].legend(fontsize=6)
    plt.tight_layout(); plt.show()

## Section 3 — TOPOLOGY (whole graph)

Connectivity of each cell treated as a **plain graph** — every node (A-pixel or B-pixel) is just
a node, sides ignored. Because the graph is strictly bipartite (edges only cross sides) it has
no triangles, so clustering / transitivity are identically 0 and are omitted; only connectivity
metrics carry information. Let `G` be the graph, `N = n_nodes`, `E = n_edges`. All metrics are
computed in `graph_phenotype.compute_topology` on the `igraph.Graph` `G` built from the
edgelist.

| modality | formula / definition | how it's computed |
|---|---|---|
| `g_density` | `2E / (N·(N−1))` — fraction of all possible node pairs joined by an edge, in [0, 1] | `G.density()` |
| `g_n_components` | number of connected components of `G` | `len(G.connected_components().sizes())` (undirected BFS/union-find over all nodes) |
| `g_largest_cc_frac` | (nodes in the largest connected component) / `N`, in [0, 1] | `max(component sizes) / N` |
| `g_average_k_core` | mean coreness over all nodes; coreness(v) = largest `k` such that `v` survives in the k-core (the maximal subgraph where every node has degree ≥ `k`) | `mean(G.coreness())` — iterative peeling: repeatedly remove the lowest-degree node, recording the degree at removal |
| `g_degree_assortativity` | Pearson correlation of the degrees at the two endpoints of each edge, in [−1, 1] | `G.assortativity_degree(directed=False)`; `nan` when `E = 0` |

**Path/diameter** (`g_avg_path_length`, `g_diameter`, Section 4) are *not* exact: all-pairs
shortest paths are infeasible on ~60k-node cells, so they run on a random node-subsample
(`PATH_SUBSAMPLE = 600`) of the largest component, then that subsample's giant component
(`giant.average_path_length()` / `.diameter()`).

### Null-ratio (`{feat}_ratio`) — how the null is built

Every metric except `g_density` (invariant under edge rewiring) is scored against a
**degree-preserving edge-swap null** (`topology_with_null`, `n_perm = 20` draws per cell):

1. Draw a rewired edgelist with `bipartite_edgeswap_sample` — repeated random edge swaps that
   keep **both side degree sequences exactly fixed** (each node keeps its degree; only the
   wiring changes).
2. Recompute the same topology metrics on each rewired graph (`want_path=False` — the expensive
   path/spectral features are skipped in the null loop).
3. Over the `n_perm` draws take the null mean `μ` and std `σ`, then report
   `{feat}_ratio = observed / μ`  (≈ 1 ⇒ the value is fully explained by the degree sequence
   alone; > 1 ⇒ excess structure) and `{feat}_z = (observed − μ) / σ`.

> Computed by `graph_phenotype.py` as `g_*` — **delete the old parquets and re-run Section 0** to
> generate them.

In [ ]:
# View A — entire cohort (observed + null-ratio).
family_overview(TOPO_FEATS + TOPO_RATIO,
                title="Section 3 · TOPOLOGY — cohort distributions (observed + null-ratio)")

In [ ]:
# View B — systems x conditions (all cell types pooled): observed, then null-ratio.
grid_by_system(TOPO_FEATS, title="TOPOLOGY · observed — systems x conditions")
grid_by_system(TOPO_RATIO, title="TOPOLOGY · null-ratio (obs / degree-preserving null) — systems x conditions")

In [ ]:
# View C — per cell type (CD8, then B), observed and null-ratio.
for ct in CELLTYPE_ORDER:
    d = cells[cells.cell_type_annot == ct]
    grid_by_system(TOPO_FEATS, df=d, title=f"TOPOLOGY · observed — {ct} cells")
    grid_by_system(TOPO_RATIO, df=d, title=f"TOPOLOGY · null-ratio — {ct} cells")

## Section 4 — PATH (whole graph, subsampled)

Geodesic (shortest-path) distances in the raw graph `G`. Exact all-pairs shortest paths are
infeasible on ~60k-node cells, so both metrics are computed on a **random node-subsample of the
largest connected component** (`PATH_SUBSAMPLE = 600` nodes, then that subsample's giant
component). Read them as an approximation of the true path structure, comparable across cells at
a fixed subsample size.

| modality | definition |
|---|---|
| `g_avg_path_length` | mean shortest-path distance over all reachable node pairs in the subsampled component |
| `g_diameter` | the longest shortest-path (maximum geodesic) in the subsampled component |

> Computed by `graph_phenotype.py` as `g_avg_path_length` / `g_diameter` — **re-run Section 0**.

In [ ]:
# View A — entire cohort.
family_overview(PATH_FEATS, title="Section 4 · PATH — cohort distributions")

In [ ]:
# View B — systems x conditions (all cell types pooled).
grid_by_system(PATH_FEATS, title="PATH — systems x conditions")

In [ ]:
# View C — per cell type (CD8, then B).
for ct in CELLTYPE_ORDER:
    grid_by_system(PATH_FEATS, df=cells[cells.cell_type_annot == ct], title=f"PATH — {ct} cells")

## Section 5 — PATCHES (local neighbourhood enrichment)

**Local Neighbourhood Enrichment (LNE)** of B-cell membrane markers — the trogocytosis readout,
a port of `pixelatorR::local_proximity` (analytical, `mode="any"`, `k=3`). Patch markers =
`CD20, CD22, CD40`. Computed once on the **full native-depth** graph, independent of any edge
budget.

**Algorithm.** `Ak = binarise((A + I)^k)` is the k-hop reachability matrix (a node's
neighbourhood includes itself). For node `i`, `obs_i` = number of nodes carrying **any** patch
marker within its k-hop neighbourhood; the expectation is computed **per side** to remove
cell-global abundance, `exp_i = f1 · degA_i + f2 · degB_i` (per-side patch-marker frequencies
`f1`, `f2` and per-side neighbourhood sizes `degA`, `degB`). Per node,
`LNE = log2( max(obs, 2) / max(exp, 2) )`.

| modality | definition |
|---|---|
| `lne_mean` | mean per-node log2 local enrichment |
| `lne_p90` | 90th-percentile enrichment (tail) |
| `lne_max` | peak local enrichment (a single focal transferred patch) |
| `lne_marker_nodes` | raw count of patch-marker nodes in the cell (depth-linked) |

In [ ]:
# View A — entire cohort.
family_overview(PATCH_FEATS, title="Section 5 · PATCHES (LNE) — cohort distributions")

In [ ]:
# View B — systems x conditions (all cell types pooled).
grid_by_system(PATCH_FEATS, title="PATCHES (LNE) — systems x conditions")

In [ ]:
# View C — per cell type (CD8, then B).
for ct in CELLTYPE_ORDER:
    grid_by_system(PATCH_FEATS, df=cells[cells.cell_type_annot == ct],
                   title=f"PATCHES (LNE) — {ct} cells")

### CD8 synapse-marker LNE — activating vs inhibitory

Two extra LNE scores, computed exactly like the trogocytosis patch score above but over the two
**B-cell immunological-synapse panels** from `synapse_analysis.ipynb` (`synapse_scores_design.md`
§3–§4), full panels:

- **activating** — the productive Signal-2 hub (`act_lne_*`): `HLA-DR-DP-DQ, HLA-DR, HLA-DQ,
  HLA-ABC, CD80, CD86, CD40, CD19, CD20, CD79a, CD54, CD58, CD50, CD102`
- **inhibitory** — the brake-raft (`inh_lne_*`): `CD274, CD273, CD32, CD72, CD305, CD22, CD66b,
  CD162`

On a **CD8** cell these flag B-side synapse markers trogocytosed onto the T membrane (`mode="any"`,
`k=3`), so both are shown **on CD8 only**, across systems × conditions.

> New `act_lne_*` / `inh_lne_*` columns (panels added in `graph_phenotype.py`) — **delete
> `results/graph_phenotype/*.parquet` and re-run Section 0** to generate them; until then
> `present()` drops them and the cell prints the regen note.

In [ ]:
# CD8 only: activating vs inhibitory B-synapse LNE, across systems x conditions (Views A + B).
ACT_FEATS = present(["act_lne_mean", "act_lne_p90", "act_lne_max", "act_lne_marker_nodes"])
INH_FEATS = present(["inh_lne_mean", "inh_lne_p90", "inh_lne_max", "inh_lne_marker_nodes"])
cd8 = cells[cells.cell_type_annot == "CD8"]

if not (ACT_FEATS and INH_FEATS):
    print("act_lne_*/inh_lne_* columns missing — delete results/graph_phenotype/*.parquet "
          "and re-run Section 0.")
else:
    print(f"CD8 cells: {len(cd8):,}")
    family_overview(ACT_FEATS, df=cd8, title="Activating B-synapse LNE (act_lne) — CD8 cohort")
    grid_by_system(ACT_FEATS, df=cd8, title="Activating B-synapse LNE — CD8, systems x conditions")
    family_overview(INH_FEATS, df=cd8, title="Inhibitory B-synapse LNE (inh_lne) — CD8 cohort")
    grid_by_system(INH_FEATS, df=cd8, title="Inhibitory B-synapse LNE — CD8, systems x conditions")

## Export — standalone HTML report

Collects the whole notebook into one self-contained HTML: every section's **markdown**
(feature-family descriptions + calculation details) interleaved with **all the figures**
(embedded as base64, so the file is portable). Code cells are hidden (`--no-input`).

Run this **last**, after every figure above has rendered, and **save the notebook first**
(`Cmd/Ctrl+S`) — `nbconvert` reads the `.ipynb` from disk, so unsaved figure outputs won't
appear in the report.

In [ ]:
# [Export report] — markdown + all figures -> one portable HTML (code hidden).
# Reads the .ipynb from disk: SAVE the notebook first, or the newest figures won't be included.
import subprocess, sys

NB  = BASE / "graph_phenotype_analysis.ipynb"
OUT = BASE / "results" / "graph_phenotype_report.html"

r = subprocess.run(
    [sys.executable, "-m", "nbconvert", "--to", "html", "--no-input",
     "--output", OUT.stem, "--output-dir", str(OUT.parent), str(NB)],
    capture_output=True, text=True)
if r.returncode == 0 and OUT.exists():
    print(f"wrote {OUT}  ({OUT.stat().st_size / 1e6:.1f} MB)")
else:
    print("nbconvert failed:\n", r.stderr or r.stdout)